# 01. Data Pull & Dataset Verification

**Football Passing Networks: Structure, Outcomes and Playing Style**

This notebook verifies the StatsBomb Open Data extraction for **FIFA World Cup 2018** and **FIFA World Cup 2022** matches.
It inspects the raw match metadata, truncated completed passing events (truncated at first substitution/red card for stable 11-player lineups), weighted directed edge lists, and team playing style metrics.

In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Paths
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("Checking data directories...")
print("Raw dir exists:", RAW_DIR.exists())
print("Processed dir exists:", PROCESSED_DIR.exists())

Checking data directories...
Raw dir exists: True
Processed dir exists: True


## 1. Load Processed Datasets

In [3]:
# Load data tables
matches_df = pd.read_parquet(RAW_DIR / "matches.parquet")
team_summary_df = pd.read_parquet(PROCESSED_DIR / "team_match_summary.parquet")
edges_df = pd.read_parquet(PROCESSED_DIR / "passing_edges.parquet")
player_stats_df = pd.read_parquet(PROCESSED_DIR / "player_match_stats.parquet")
passes_df = pd.read_parquet(PROCESSED_DIR / "passing_events.parquet")

print(f"Loaded {len(matches_df)} matches across competitions.")
print(f"Loaded {len(team_summary_df)} team-match summaries.")
print(f"Loaded {len(edges_df)} passing graph edges.")
print(f"Loaded {len(player_stats_df)} player match position entries.")
print(f"Loaded {len(passes_df)} completed passing events.")

Loaded 128 matches across competitions.
Loaded 256 team-match summaries.
Loaded 18924 passing graph edges.
Loaded 2810 player match position entries.
Loaded 65963 completed passing events.


## 2. Dataset Summary & Match Outcomes

In [4]:
print("--- Matches Breakdown by Season ---")
print(matches_df["season_name"].value_counts())

print("\n--- Team Match Results (Win / Loss / Draw) ---")
print(team_summary_df["result"].value_counts())

display(team_summary_df.head(10))

--- Matches Breakdown by Season ---
season_name
2018    64
2022    64
Name: count, dtype: int64

--- Team Match Results (Win / Loss / Draw) ---
result
Loss    100
Win     100
Draw     56
Name: count, dtype: int64


,match_id,competition_name,season_name,team_name,opponent_name,result,goals_for,goals_against,truncated_pass_count,avg_pass_length,progressive_pass_ratio,first_sub_minute
0,8650,FIFA World Cup,2018,Brazil,Belgium,Loss,1,2,226,17.093478,0.190265,45
1,8650,FIFA World Cup,2018,Belgium,Brazil,Win,2,1,279,21.687929,0.290323,82
2,7584,FIFA World Cup,2018,Belgium,Japan,Win,3,2,380,19.391534,0.231579,64
3,7584,FIFA World Cup,2018,Japan,Belgium,Loss,2,3,313,19.452051,0.230032,80
4,7554,FIFA World Cup,2018,England,Panama,Win,6,1,364,18.132745,0.200549,62
5,7554,FIFA World Cup,2018,Panama,England,Loss,1,6,199,19.559691,0.226131,61
6,7539,FIFA World Cup,2018,Poland,Senegal,Loss,1,2,235,21.408539,0.251064,45
7,7539,FIFA World Cup,2018,Senegal,Poland,Win,2,1,209,19.889825,0.196172,61
8,7550,FIFA World Cup,2018,Serbia,Switzerland,Loss,1,2,157,24.451820,0.369427,63
9,7550,FIFA World Cup,2018,Switzerland,Serbia,Win,2,1,274,19.859375,0.229927,45


## 3. Sample Passing Edge List & Graph Construction

In [5]:
# Select a sample team-match graph (e.g., Argentina in a match)
sample_match_id = team_summary_df.iloc[0]["match_id"]
sample_team = team_summary_df.iloc[0]["team_name"]

sample_edges = edges_df[(edges_df["match_id"] == sample_match_id) & (edges_df["team_name"] == sample_team)]
print(f"Sample Passing Edges for {sample_team} (Match ID: {sample_match_id}):")
display(sample_edges.head(10))

# Build NetworkX DiGraph
G = nx.DiGraph()
for _, row in sample_edges.iterrows():
    w = row["weight"]
    G.add_edge(row["passer_name"], row["recipient_name"], weight=w, distance=1.0 / w)

print(f"\nNetworkX Graph Built: {G.number_of_nodes()} nodes (players), {G.number_of_edges()} directed edges.")

Sample Passing Edges for Brazil (Match ID: 8650):


,match_id,competition_name,season_name,team_name,passer_id,passer_name,recipient_id,recipient_name,weight,avg_distance,progressive_passes
0,8650,FIFA World Cup,2018,Brazil,3054,Fernando Luiz Roza,3295,Thiago Emiliano da Silva,2,12.322580,0
1,8650,FIFA World Cup,2018,Brazil,3054,Fernando Luiz Roza,3472,Willian Borges da Silva,5,25.013333,2
2,8650,FIFA World Cup,2018,Brazil,3054,Fernando Luiz Roza,3501,Philippe Coutinho Correia,6,15.310131,1
3,8650,FIFA World Cup,2018,Brazil,3054,Fernando Luiz Roza,4320,Neymar da Silva Santos Junior,1,29.068884,0
4,8650,FIFA World Cup,2018,Brazil,3054,Fernando Luiz Roza,5542,José Paulo Bezzera Maciel Júnior,6,12.506152,1
5,8650,FIFA World Cup,2018,Brazil,3054,Fernando Luiz Roza,5551,João Miranda de Souza Filho,2,5.612268,0
6,8650,FIFA World Cup,2018,Brazil,3054,Fernando Luiz Roza,5552,Marcelo Vieira da Silva Júnior,2,19.525641,0
7,8650,FIFA World Cup,2018,Brazil,3054,Fernando Luiz Roza,5838,Fágner Conserva Lemos,2,16.569077,0
8,8650,FIFA World Cup,2018,Brazil,3202,Gabriel Fernando de Jesus,3501,Philippe Coutinho Correia,1,10.630146,0
9,8650,FIFA World Cup,2018,Brazil,3202,Gabriel Fernando de Jesus,4320,Neymar da Silva Santos Junior,1,11.661903,0



NetworkX Graph Built: 11 nodes (players), 64 directed edges.
